# Sequitur Algorithm

Sequitur (Nevill-Manning & Witten, 1997) is an online algorithm that infers a **context-free grammar** from a flat sequence of symbols in **linear time and space**. It discovers hierarchical, repeated structure by maintaining just two invariants:

1. **Digram uniqueness** — no pair of adjacent symbols appears more than once across all rules.
2. **Rule utility** — every rule (except the start rule) is referenced at least twice.

These two constraints work in tension: digram uniqueness *creates* rules, rule utility *removes* them, and together they keep the grammar minimal.

## Implementation

Each rule's right-hand side is a doubly-linked list of symbols, giving O(1) splicing.
A hash table indexes every digram in the grammar for O(1) duplicate detection.

In [16]:
class Symbol:
    """A node in a rule's doubly-linked list. Holds either a terminal (str) or a rule reference."""

    def __init__(self, value):
        self.value = value
        self.prev = None
        self.next = None

    @property
    def is_guard(self):
        return False

    def key(self):
        return self.value if isinstance(self.value, str) else id(self.value)

    def insert_after(self, sym):
        sym.prev = self
        sym.next = self.next
        self.next.prev = sym
        self.next = sym

    def unlink(self):
        self.prev.next = self.next
        self.next.prev = self.prev

    def __repr__(self):
        if isinstance(self.value, str):
            return self.value
        return str(self.value)


class Guard(Symbol):
    """Sentinel that bookends a rule's linked list."""

    def __init__(self, rule):
        super().__init__(rule)
        self.prev = self
        self.next = self

    @property
    def is_guard(self):
        return True


class Rule:
    _counter = 0

    def __init__(self):
        self.guard = Guard(self)
        self.ref_count = 0
        self.index = Rule._counter
        Rule._counter += 1

    def first(self):
        return self.guard.next

    def last(self):
        return self.guard.prev

    def symbols(self):
        s = self.guard.next
        while not s.is_guard:
            yield s
            s = s.next

    def __len__(self):
        return sum(1 for _ in self.symbols())

    def __repr__(self):
        return 'S' if self.index == 0 else chr(ord('A') + (self.index - 1) % 26)

    def format(self):
        return f"{self} \u2192 {' '.join(repr(s) for s in self.symbols())}"

In [17]:
def _digram_key(a, b):
    return (a.key(), b.key())


class Sequitur:
    """Builds a context-free grammar from a stream of characters."""

    def __init__(self):
        Rule._counter = 0
        self.start = Rule()
        self.digrams = {}

    # -- public API ------------------------------------------------------------

    def feed(self, text):
        for ch in text:
            self._append(ch)

    def rules(self):
        seen = set()
        queue = [self.start]
        while queue:
            rule = queue.pop(0)
            if id(rule) in seen:
                continue
            seen.add(id(rule))
            yield rule
            for sym in rule.symbols():
                if isinstance(sym.value, Rule):
                    queue.append(sym.value)

    def expand(self, rule=None):
        if rule is None:
            rule = self.start
        parts = []
        for sym in rule.symbols():
            if isinstance(sym.value, str):
                parts.append(sym.value)
            else:
                parts.append(self.expand(sym.value))
        return ''.join(parts)

    # -- internals -------------------------------------------------------------

    def _append(self, ch):
        new_sym = Symbol(ch)
        self.start.last().insert_after(new_sym)
        self._check_digram(new_sym.prev)

    def _check_digram(self, sym):
        """Enforce digram uniqueness at the pair (sym, sym.next)."""
        if sym.is_guard or sym.next.is_guard:
            return

        key = _digram_key(sym, sym.next)
        match = self.digrams.get(key)

        if match is None:
            self.digrams[key] = sym
            return

        if match is sym or match is sym.next:
            return

        # Overlapping digrams (e.g. X X X): the two pairs share a middle
        # symbol and cannot be safely substituted.  The original paper
        # (Nevill-Manning & Witten, 1997 §4.3) explicitly skips this case.
        if match.next is sym:
            return

        # Duplicate found.
        if match.prev.is_guard and match.next.next.is_guard:
            # The match IS an entire rule body -> reuse that rule.
            rule = match.prev.value
            self._substitute(sym, rule)
        else:
            # Create a new rule from the matched pair.
            rule = Rule()
            first = Symbol(match.value)
            second = Symbol(match.next.value)
            rule.guard.insert_after(first)
            first.insert_after(second)
            if isinstance(first.value, Rule):
                first.value.ref_count += 1
            if isinstance(second.value, Rule):
                second.value.ref_count += 1

            self._substitute(match, rule)
            self._substitute(sym, rule)

            self.digrams[_digram_key(rule.first(), rule.first().next)] = rule.first()

    def _substitute(self, sym, rule):
        """Replace the digram (sym, sym.next) with a reference to rule."""
        right = sym.next

        # Unregister neighbouring digrams that will change.
        self._unregister(sym)
        if not right.next.is_guard:
            self._unregister(right)
        if not sym.prev.is_guard:
            self._unregister(sym.prev)

        # Remove the right symbol.
        right.unlink()
        right_val = right.value
        if isinstance(right_val, Rule):
            right_val.ref_count -= 1

        # Replace the left symbol's value with the rule reference.
        left_val = sym.value
        if isinstance(left_val, Rule):
            left_val.ref_count -= 1
        sym.value = rule
        rule.ref_count += 1

        # Enforce rule utility on any rules whose ref count just dropped.
        if isinstance(left_val, Rule) and left_val.ref_count == 1:
            self._inline(left_val)
        if isinstance(right_val, Rule) and right_val.ref_count == 1:
            self._inline(right_val)

        # Register new digrams formed at the edges.
        if not sym.prev.is_guard:
            self._check_digram(sym.prev)
        if not sym.next.is_guard:
            self._check_digram(sym)

    def _inline(self, rule):
        """A rule is used only once — expand it inline and delete."""
        ref = self._find_reference(rule)
        if ref is None:
            return

        first = rule.first()
        last = rule.last()

        # Unregister digrams around the reference site.
        if not ref.prev.is_guard:
            self._unregister(ref.prev)
        if not ref.next.is_guard:
            self._unregister(ref)

        # Splice the rule body in place of the reference.
        ref.prev.next = first
        first.prev = ref.prev
        ref.next.prev = last
        last.next = ref.next

        # Disconnect the rule guard so it's cleanly deleted.
        rule.guard.next = rule.guard
        rule.guard.prev = rule.guard

        # Register join-point digrams (don't recurse — just register).
        if not first.prev.is_guard:
            self.digrams[_digram_key(first.prev, first)] = first.prev
        if not last.next.is_guard:
            self.digrams[_digram_key(last, last.next)] = last

    def _find_reference(self, rule):
        """Find the single Symbol whose value is *rule*."""
        for r in self.rules():
            for sym in r.symbols():
                if sym.value is rule:
                    return sym
        return None

    def _unregister(self, sym):
        if sym.is_guard or sym.next.is_guard:
            return
        key = _digram_key(sym, sym.next)
        if self.digrams.get(key) is sym:
            del self.digrams[key]

    def __repr__(self):
        return '\n'.join(r.format() for r in self.rules())

## Example: Step by Step

Let's trace the algorithm on a small string to see rules form.

In [18]:
seq = Sequitur()
text = "abcabcabc"

for i, ch in enumerate(text, 1):
    seq.feed(ch)
    print(f"After reading '{text[:i]}':")
    print(f"  {seq}")
    print()

After reading 'a':
  S → a

After reading 'ab':
  S → a b

After reading 'abc':
  S → a b c

After reading 'abca':
  S → a b c a

After reading 'abcab':
  S → A c A
A → a b

After reading 'abcabc':
  S → B B
B → a b c

After reading 'abcabca':
  S → B B a
B → a b c

After reading 'abcabcab':
  S → B B C
B → C c
C → a b

After reading 'abcabcabc':
  S → B B B
B → a b c



In [19]:
# Verify the grammar expands back to the original string.
assert seq.expand() == text, f"Expected {text!r}, got {seq.expand()!r}"
print(f"Original : {text}")
print(f"Expanded : {seq.expand()}")
print(f"\nFinal grammar:")
print(seq)

Original : abcabcabc
Expanded : abcabcabc

Final grammar:
S → B B B
B → a b c


## Longer Example

A longer, more repetitive string shows deeper hierarchical structure.

In [20]:
seq2 = Sequitur()
longer = "abcdabcdabcdabcd"
seq2.feed(longer)

assert seq2.expand() == longer
print(f"Input ({len(longer)} chars): {longer}")
print(f"\nGrammar:")
print(seq2)
print(f"\nExpands to: {seq2.expand()}")

Input (16 chars): abcdabcdabcdabcd

Grammar:
S → H H
H → C C
C → a b c d

Expands to: abcdabcdabcdabcd


In [21]:
# Counting the total symbols in the grammar vs the original length.
grammar_symbols = sum(len(r) for r in seq2.rules())
print(f"Original length   : {len(longer)}")
print(f"Grammar symbols   : {grammar_symbols}")
print(f"Compression ratio : {grammar_symbols / len(longer):.2f}")

Original length   : 16
Grammar symbols   : 8
Compression ratio : 0.50


## How It Works — Recap

1. **Scan left to right.** Each new character is appended to the start rule S.
2. **Check the new digram** (the last two symbols of S) against a hash table.
   - If unseen, register it.
   - If already seen, either **reuse** the existing rule whose body matches, or **create** a new rule and replace both occurrences.
3. **Enforce rule utility.** If any rule's reference count drops to 1, inline it and delete.
4. Repeat until the input is consumed.

The result is a grammar that generates exactly the input string, with repeated substrings factored into reusable rules — all in **O(n) time and space**.

## References

- Nevill-Manning, C.G. and Witten, I.H. (1997). **"Identifying Hierarchical Structure in Sequences: A linear-time algorithm."** *Journal of Artificial Intelligence Research*, 7, 67–82. [https://doi.org/10.1613/jair.374](https://doi.org/10.1613/jair.374)
- [Wikipedia: Sequitur algorithm](https://en.wikipedia.org/wiki/Sequitur_algorithm)